In [ ]:
# Import Libraries
import pandas as pd
import exploration.explore as ex
import matplotlib as plot
import re

In [ ]:
pd.options.display.max_columns = 30

## Goals
- Create a combined dataset, and note any areas where joins will be difficult / will require logic
- Find a variable to use as our outcome - did a bill pass - and note different thresholds we might use
- Explore general shape of the data, e.g. how many bills are introduced, how many pass, breakdown by bill type
- Find and create clear features to use to inform 

### Goal 1: Create a combined datset (and note edge cases)

In [ ]:
# Get all data associated with Bills
all_data = ex.get_all_datasets()

In [ ]:
# Base data
bills = all_data["bills"]

bill_abstracts = all_data["bill_abstracts"]
bill_actions = all_data["actions"]
bills_sponsorships = all_data["bill_sponsorships"]
bill_organizations = all_data["organizations"]
bill_vote_counts = all_data["vote_counts"]
bill_votes = all_data["votes"]
bill_vote_people = all_data["vote_people"]

In [ ]:
files = (bill_abstracts,
         bill_actions,
         bills_sponsorships,
         bills,
         bill_organizations,
         bill_vote_counts,
         bill_votes,
         bill_vote_people)

In [ ]:
bills.shape  # (22783, 8) - the number of rows checks out

(26222, 8)

In [ ]:
bills.head()

,id,identifier,title,classification,subject,session_identifier,jurisdiction,organization_classification
0,ocd-bill/f4c0d518-f592-4828-8c23-532ec44ed642,HB 1277,PEN CD-FELONY-SUSPEND BENEFITS,['bill'],[],103rd,Illinois,lower
1,ocd-bill/ea6533c0-942f-4f92-aef7-bef03046387c,HB 4017,$DHS-INDEPENDENT LIVING CNTRS,['bill'],[],103rd,Illinois,lower
2,ocd-bill/0372f02d-3773-4ce5-9f26-26f0141e2227,SB 2535,YOUTH NON-VIOLENT RESOURCES,['bill'],[],103rd,Illinois,upper
3,ocd-bill/90c3fdab-a08d-4e41-aac0-41fb9b462ea6,HR 376,CONGRATS-MARK AND ROSE FROSETH,['resolution'],[],103rd,Illinois,lower
4,ocd-bill/d7335ae9-ecc0-41e9-8e89-94435aa00e59,HR 375,MEMORIAL-ANTHONY RUSSELL,['resolution'],[],103rd,Illinois,lower


In [ ]:
# Merge Bills and Bill Abstracts
df = bills.merge(bill_abstracts, how = "inner", left_on = "id", right_on = "bill_id")  # (22782, 12)

In [ ]:
# Fix id column naming (not sure why it changed to "id_x")
df = df.rename(columns = {"id_x": "id"})

In [ ]:
df = df[df["classification"].apply(lambda x: "bill" in x)]

In [ ]:
df.head()

,id,identifier,title,classification,subject,session_identifier,jurisdiction,organization_classification,id_y,bill_id,abstract,note
0,ocd-bill/f4c0d518-f592-4828-8c23-532ec44ed642,HB 1277,PEN CD-FELONY-SUSPEND BENEFITS,['bill'],[],103rd,Illinois,lower,014fdfc4-5501-4ad3-8e09-2148cc32aceb,ocd-bill/f4c0d518-f592-4828-8c23-532ec44ed642,Amends the General Provisions Article of the I...,NaN
1,ocd-bill/ea6533c0-942f-4f92-aef7-bef03046387c,HB 4017,$DHS-INDEPENDENT LIVING CNTRS,['bill'],[],103rd,Illinois,lower,e57822d9-cd63-48ab-a3df-f02d0d29d8bf,ocd-bill/ea6533c0-942f-4f92-aef7-bef03046387c,"Appropriates $16,358,900 from the General Reve...",NaN
2,ocd-bill/0372f02d-3773-4ce5-9f26-26f0141e2227,SB 2535,YOUTH NON-VIOLENT RESOURCES,['bill'],[],103rd,Illinois,upper,adb655ba-af0b-430d-a932-6f91043f3cac,ocd-bill/0372f02d-3773-4ce5-9f26-26f0141e2227,Amends the Illinois Criminal Justice Informati...,NaN
5,ocd-bill/b3f19d64-64ee-4061-94be-294fd4d5b6ca,SB 2816,ORGANIC WASTE COMPOSTING,['bill'],[],103rd,Illinois,upper,408e444f-c63f-4484-aacc-f63969f9f82d,ocd-bill/b3f19d64-64ee-4061-94be-294fd4d5b6ca,Amends the Environmental Protection Act. Requi...,NaN
6,ocd-bill/4ae56f70-b57b-4d05-a6ef-d22839d5fee1,HB 3418,REENTRY INTO THE WORKFORCE,['bill'],[],103rd,Illinois,lower,da4fc3ab-ebdf-428b-bc42-7fd2a25829c1,ocd-bill/4ae56f70-b57b-4d05-a6ef-d22839d5fee1,Creates the Securing All Futures through Equit...,NaN


In [ ]:
bill_actions.head()

,id,bill_id,organization_id,description,date,classification,order
0,40cd4dfc-32d3-4b26-84c6-8cae63f6f353,ocd-bill/64fcf73e-0478-4648-b6e1-da08c7d38e41,ocd-organization/76120f9a-2982-4cc5-a6e2-de740...,"Prefiled with Clerk by Rep. Emanuel ""Chris"" Welch",2022-12-05,['filing'],0
1,30006f62-6165-4507-9e07-317a5b413895,ocd-bill/0c271228-af51-4133-a953-8fc6ae2f4749,ocd-organization/76120f9a-2982-4cc5-a6e2-de740...,Filed with the Clerk by Rep. Sue Scherer,2024-01-18,['filing'],0
2,64991aee-01dd-4670-b5bf-b81e839c7fe4,ocd-bill/883f0752-a648-4114-bbb6-88ca03678562,ocd-organization/76120f9a-2982-4cc5-a6e2-de740...,Filed with the Clerk by Rep. Tony M. McCombie,2023-02-01,['filing'],0
3,4d7d91fc-be39-402b-b38f-464e72bae898,ocd-bill/395afc7c-c1bf-4a05-8e62-06f229e96292,ocd-organization/76120f9a-2982-4cc5-a6e2-de740...,"Prefiled with Clerk by Rep. Emanuel ""Chris"" Welch",2022-12-06,['filing'],0
4,b933f9dd-356d-4cb9-a6e9-ea1345556fdb,ocd-bill/5e1bed6d-cf60-46fd-84f0-ff00cde573d2,ocd-organization/76120f9a-2982-4cc5-a6e2-de740...,Filed with the Clerk by Rep. Tony M. McCombie,2023-02-01,['filing'],0


In [ ]:
bill_actions["classification"].value_counts(normalize=True) * 100

classification
[]                                 48.147481
['referral-committee']             14.342887
['filing']                          7.990194
['reading-1']                       7.688173
['reading-2']                       6.514579
['failure']                         2.902135
['amendment-introduction']          1.931132
['committee-passage']               1.670126
['passage', 'reading-3']            1.633150
['passage']                         1.463807
['amendment-passage']               1.147182
['introduction']                    1.127296
['executive-receipt']               0.687626
['became-law']                      0.684208
['executive-signature']             0.677061
['committee-passage-favorable']     0.669293
['reading-3']                       0.441535
['amendment-failure']               0.259763
['committee-failure']               0.011807
['withdrawal']                      0.004350
['executive-veto']                  0.004350
['failure', 'reading-3']            0.00

In [ ]:
bill_actions.groupby("bill_id")["date"].count().sort_values(ascending=False).head(20)

bill_id
ocd-bill/1cfbb486-1332-45ed-8438-9bc174dd6a8d    187
ocd-bill/019153d8-a121-4414-bcbc-f9a4a2b4dd86    185
ocd-bill/3578f1c6-52d2-4d2d-a560-9724928645a9    182
ocd-bill/8dab24a3-e0b9-43b3-aecf-10590e860f1c    181
ocd-bill/dd3a42df-2f5a-4848-a2a7-0851def7b7fd    179
ocd-bill/dc8709ca-31cc-4498-b40c-7b7344789679    173
ocd-bill/a7200893-bcb1-4467-aeba-dd5f85b5862b    168
ocd-bill/6ee5b9bc-d941-4350-9437-6c9dcd4c7502    164
ocd-bill/70380a50-000c-429b-a618-18c932ab7016    162
ocd-bill/e8e896d5-fd75-4a11-b6fd-fa1968bf411b    157
ocd-bill/f14310d4-6157-47d6-87ee-388d67767ddf    154
ocd-bill/85078383-95d5-4494-8b41-f7fec1b425d9    151
ocd-bill/90d07fbb-9f46-441e-b692-9238a6b3352b    149
ocd-bill/38f967ba-9287-4f34-841c-35e72e443c94    148
ocd-bill/95592f1f-3f54-48c1-857d-65ced14cc7e8    145
ocd-bill/c974da52-b390-4907-930a-aee431d609c0    142
ocd-bill/8e096bc4-9be7-4ba3-9e7c-1d6541dd804d    141
ocd-bill/7eca898c-e95e-410a-8292-1edd8d3fd794    140
ocd-bill/e503e1c7-e87b-4fda-9627-872e2

In [ ]:
df[df["id"] == "ocd-bill/6be183d7-5d16-4b38-9247-7c021f01d6ec"]

,id,identifier,title,classification,subject,session_identifier,jurisdiction,organization_classification,id_y,bill_id,abstract,note


In [ ]:
bill_actions[bill_actions["bill_id"] == "ocd-bill/9314b947-01a7-4783-9a12-db06d43f3b4"]

,id,bill_id,organization_id,description,date,classification,order


In [ ]:
num_bills_w_actions = bill_actions["bill_id"].nunique() # 22,783 Bills have unique actions
num_bills_w_actions

26222

In [ ]:
mask = bill_actions["description"] == "Passed Both Houses"
num_bills_pass_both = bill_actions[mask]

In [ ]:
num_bills_pass_both["bill_id"].nunique()

2224

In [ ]:
mask2 = bill_actions["description"] == "Sent to the Governor"
num_bills_exec_received = bill_actions[mask2]

In [ ]:
num_bills_exec_received["bill_id"].nunique()

2212

In [ ]:
bill_actions["classification"].apply(lambda x: "executive-receipt" in x).sum()

np.int64(2213)

In [ ]:
bill_actions["classification"].apply(lambda x: "became-law" in x).sum()

np.int64(2202)

In [ ]:
bill_actions["classification"].apply(lambda x: "veto" in x).sum()

np.int64(16)

In [ ]:
bill_actions.head(10)

,id,bill_id,organization_id,description,date,classification,order
0,40cd4dfc-32d3-4b26-84c6-8cae63f6f353,ocd-bill/64fcf73e-0478-4648-b6e1-da08c7d38e41,ocd-organization/76120f9a-2982-4cc5-a6e2-de740...,"Prefiled with Clerk by Rep. Emanuel ""Chris"" Welch",2022-12-05,['filing'],0
1,30006f62-6165-4507-9e07-317a5b413895,ocd-bill/0c271228-af51-4133-a953-8fc6ae2f4749,ocd-organization/76120f9a-2982-4cc5-a6e2-de740...,Filed with the Clerk by Rep. Sue Scherer,2024-01-18,['filing'],0
2,64991aee-01dd-4670-b5bf-b81e839c7fe4,ocd-bill/883f0752-a648-4114-bbb6-88ca03678562,ocd-organization/76120f9a-2982-4cc5-a6e2-de740...,Filed with the Clerk by Rep. Tony M. McCombie,2023-02-01,['filing'],0
3,4d7d91fc-be39-402b-b38f-464e72bae898,ocd-bill/395afc7c-c1bf-4a05-8e62-06f229e96292,ocd-organization/76120f9a-2982-4cc5-a6e2-de740...,"Prefiled with Clerk by Rep. Emanuel ""Chris"" Welch",2022-12-06,['filing'],0
4,b933f9dd-356d-4cb9-a6e9-ea1345556fdb,ocd-bill/5e1bed6d-cf60-46fd-84f0-ff00cde573d2,ocd-organization/76120f9a-2982-4cc5-a6e2-de740...,Filed with the Clerk by Rep. Tony M. McCombie,2023-02-01,['filing'],0
5,ae5fd95d-4f2e-4d30-9588-e7ea551c4a42,ocd-bill/328d2ce5-f73c-424e-96b8-d1f0a3426588,ocd-organization/76120f9a-2982-4cc5-a6e2-de740...,Filed with the Clerk by Rep. Jeff Keicher,2023-02-14,['filing'],0
6,16a00f2e-58b7-41a3-9383-4919570013cc,ocd-bill/075ff76f-20c8-4850-84b8-6ade3fda658c,ocd-organization/76120f9a-2982-4cc5-a6e2-de740...,Filed with the Clerk by Rep. Tony M. McCombie,2023-02-01,['filing'],0
7,a97b60e9-16a2-42e6-a156-6481da68818e,ocd-bill/c9d53d00-87a1-47bf-968b-5d824a057a3e,ocd-organization/76120f9a-2982-4cc5-a6e2-de740...,Filed with the Clerk by Rep. Daniel Didech,2023-02-02,['filing'],0
8,dc7aac17-24a1-4257-ac03-fabdd6f06e0b,ocd-bill/82489c2a-d3b9-4c00-a40e-481db83d0926,ocd-organization/76120f9a-2982-4cc5-a6e2-de740...,Received by the Senate Sen. Laura M. Murphy,2024-01-10,[],0
9,3b75e03d-5acc-4ff5-bf64-dbab20b64ecf,ocd-bill/02b86444-4e23-4b2d-812b-2e4ccf0519a9,ocd-organization/76120f9a-2982-4cc5-a6e2-de740...,"Prefiled with Clerk by Rep. Emanuel ""Chris"" Welch",2022-12-05,['filing'],0


In [ ]:
bill_actions.dtypes

id                   str
bill_id              str
organization_id      str
description          str
date                 str
classification       str
order              int64
dtype: object

In [ ]:
bill_actions['classification'].value_counts()

classification
[]                                 154954
['referral-committee']              46160
['filing']                          25715
['reading-1']                       24743
['reading-2']                       20966
['failure']                          9340
['amendment-introduction']           6215
['committee-passage']                5375
['passage', 'reading-3']             5256
['passage']                          4711
['amendment-passage']                3692
['introduction']                     3628
['executive-receipt']                2213
['became-law']                       2202
['executive-signature']              2179
['committee-passage-favorable']      2154
['reading-3']                        1421
['amendment-failure']                 836
['committee-failure']                  38
['withdrawal']                         14
['executive-veto']                     14
['failure', 'reading-3']                4
['executive-veto-line-item']            1
['veto-override-pas

In [ ]:
bill_actions.groupby("bill_id")["classification"].apply(lambda x: x.apply(lambda classif: "amendment-introduction" in classif).any()).sum()

np.int64(3537)

In [ ]:
bill_actions.groupby("bill_id")["classification"].apply(lambda x: x.apply(lambda classif: "committee-passage" in classif).any()).sum()

np.int64(5441)

In [ ]:
bill_actions.groupby("bill_id")["classification"].apply(lambda x: x.apply(lambda classif: "reading" in classif).sum())

bill_id
ocd-bill/00001671-dd8d-4867-9572-c8a250541072    1
ocd-bill/0003dba0-925c-4f22-8e6c-2b9e10ed5c71    1
ocd-bill/000c89e5-2a87-470f-b478-601e14d57c5e    0
ocd-bill/00107fd4-1658-4ff0-ace8-a4cc245cee80    1
ocd-bill/00120e78-5b0f-4d45-9705-abd27260421e    0
                                                ..
ocd-bill/fff5d8af-80b1-4a98-a977-93fea7fea565    1
ocd-bill/fff7f8e7-1090-414c-b58f-da3d4dd4c1b7    0
ocd-bill/fff895cc-96a2-488a-b583-e73af4ae1bee    1
ocd-bill/fff9fcd7-659c-4aeb-a80a-bc256af05841    3
ocd-bill/fffef98f-1afb-44c1-88ca-f5f3181d8be8    1
Name: classification, Length: 26222, dtype: int64

In [ ]:
bill_actions.groupby("bill_id")["classification"].apply(lambda x: x.apply(lambda classif: "reading-1" in classif))

bill_id                                              
ocd-bill/00001671-dd8d-4867-9572-c8a250541072  12806     False
                                               14464      True
                                               33297     False
ocd-bill/0003dba0-925c-4f22-8e6c-2b9e10ed5c71  6133      False
                                               24331      True
                                                         ...  
ocd-bill/fff9fcd7-659c-4aeb-a80a-bc256af05841  78742     False
ocd-bill/fffef98f-1afb-44c1-88ca-f5f3181d8be8  164710    False
                                               181181     True
                                               187043    False
                                               204242    False
Name: classification, Length: 321832, dtype: bool

In [ ]:
bill_actions.groupby("bill_id")["classification"].apply(lambda x: x.apply(lambda classif: "reading-1" in classif).any())

bill_id
ocd-bill/00001671-dd8d-4867-9572-c8a250541072     True
ocd-bill/0003dba0-925c-4f22-8e6c-2b9e10ed5c71     True
ocd-bill/000c89e5-2a87-470f-b478-601e14d57c5e    False
ocd-bill/00107fd4-1658-4ff0-ace8-a4cc245cee80     True
ocd-bill/00120e78-5b0f-4d45-9705-abd27260421e    False
                                                 ...  
ocd-bill/fff5d8af-80b1-4a98-a977-93fea7fea565     True
ocd-bill/fff7f8e7-1090-414c-b58f-da3d4dd4c1b7    False
ocd-bill/fff895cc-96a2-488a-b583-e73af4ae1bee     True
ocd-bill/fff9fcd7-659c-4aeb-a80a-bc256af05841     True
ocd-bill/fffef98f-1afb-44c1-88ca-f5f3181d8be8     True
Name: classification, Length: 26222, dtype: bool

In [ ]:
bill_actions[bill_actions["bill_id"] == "ocd-bill/d2aaa200-839d-41c5-a69f-1dda8c50e67e"].to_clipboard()

In [ ]:
bill_actions["date"] = pd.to_datetime(bill_actions["date"])

In [ ]:
# Working!
first_reading = bill_actions[bill_actions["classification"].apply(lambda x: "reading-1" in x)][["bill_id", "date"]]
first_reading.groupby("bill_id")["date"].min()

bill_id
ocd-bill/00001671-dd8d-4867-9572-c8a250541072   2023-02-09
ocd-bill/0003dba0-925c-4f22-8e6c-2b9e10ed5c71   2023-02-17
ocd-bill/00107fd4-1658-4ff0-ace8-a4cc245cee80   2023-02-09
ocd-bill/0019b80b-8a8e-4dbd-82ce-a5a14a35c85f   2022-01-14
ocd-bill/001cd622-4e52-4164-bb2a-9c3d40cbd62d   2021-02-24
                                                   ...    
ocd-bill/fff3e4ad-6b2f-4f90-8daf-00099d077be0   2023-02-02
ocd-bill/fff5d8af-80b1-4a98-a977-93fea7fea565   2022-01-18
ocd-bill/fff895cc-96a2-488a-b583-e73af4ae1bee   2024-02-09
ocd-bill/fff9fcd7-659c-4aeb-a80a-bc256af05841   2023-02-02
ocd-bill/fffef98f-1afb-44c1-88ca-f5f3181d8be8   2021-02-26
Name: date, Length: 20046, dtype: datetime64[us]

In [ ]:
first_referral = bill_actions[bill_actions["classification"].apply(lambda x: "referral-committee" in x)][["bill_id", "date"]]
first_referral.groupby("bill_id")["date"].min()

bill_id
ocd-bill/000c89e5-2a87-470f-b478-601e14d57c5e   2021-05-04
ocd-bill/00107fd4-1658-4ff0-ace8-a4cc245cee80   2023-02-21
ocd-bill/0019b80b-8a8e-4dbd-82ce-a5a14a35c85f   2022-01-14
ocd-bill/001cd622-4e52-4164-bb2a-9c3d40cbd62d   2021-02-24
ocd-bill/001df1a2-95b4-4a7b-ae0b-2ea7300b5ec4   2021-01-22
                                                   ...    
ocd-bill/fff3e4ad-6b2f-4f90-8daf-00099d077be0   2023-03-02
ocd-bill/fff46e55-9697-476d-82f6-1305d99b7211   2023-03-07
ocd-bill/fff5d8af-80b1-4a98-a977-93fea7fea565   2022-01-18
ocd-bill/fff9fcd7-659c-4aeb-a80a-bc256af05841   2023-03-02
ocd-bill/fffef98f-1afb-44c1-88ca-f5f3181d8be8   2021-02-26
Name: date, Length: 20159, dtype: datetime64[us]

In [ ]:
len(bill_actions.groupby("bill_id"))

26222

In [ ]:
# bill_actions.groupby("bill_id")[bill_actions["classification"].apply(lambda x: "reading-1" in x)]["date"]

### Create Summary Bill Actions Table

In [ ]:
bill_actions.head()

,id,bill_id,organization_id,description,date,classification,order
0,40cd4dfc-32d3-4b26-84c6-8cae63f6f353,ocd-bill/64fcf73e-0478-4648-b6e1-da08c7d38e41,ocd-organization/76120f9a-2982-4cc5-a6e2-de740...,"Prefiled with Clerk by Rep. Emanuel ""Chris"" Welch",2022-12-05,['filing'],0
1,30006f62-6165-4507-9e07-317a5b413895,ocd-bill/0c271228-af51-4133-a953-8fc6ae2f4749,ocd-organization/76120f9a-2982-4cc5-a6e2-de740...,Filed with the Clerk by Rep. Sue Scherer,2024-01-18,['filing'],0
2,64991aee-01dd-4670-b5bf-b81e839c7fe4,ocd-bill/883f0752-a648-4114-bbb6-88ca03678562,ocd-organization/76120f9a-2982-4cc5-a6e2-de740...,Filed with the Clerk by Rep. Tony M. McCombie,2023-02-01,['filing'],0
3,4d7d91fc-be39-402b-b38f-464e72bae898,ocd-bill/395afc7c-c1bf-4a05-8e62-06f229e96292,ocd-organization/76120f9a-2982-4cc5-a6e2-de740...,"Prefiled with Clerk by Rep. Emanuel ""Chris"" Welch",2022-12-06,['filing'],0
4,b933f9dd-356d-4cb9-a6e9-ea1345556fdb,ocd-bill/5e1bed6d-cf60-46fd-84f0-ff00cde573d2,ocd-organization/76120f9a-2982-4cc5-a6e2-de740...,Filed with the Clerk by Rep. Tony M. McCombie,2023-02-01,['filing'],0


In [ ]:
bill_actions.groupby("bill_id")["date"].agg(["max", "max"])

,max,max
bill_id,,
ocd-bill/00001671-dd8d-4867-9572-c8a250541072,2023-02-09,2023-02-09
ocd-bill/0003dba0-925c-4f22-8e6c-2b9e10ed5c71,2023-02-17,2023-02-17
ocd-bill/000c89e5-2a87-470f-b478-601e14d57c5e,2023-01-10,2023-01-10
ocd-bill/00107fd4-1658-4ff0-ace8-a4cc245cee80,2023-03-10,2023-03-10
ocd-bill/00120e78-5b0f-4d45-9705-abd27260421e,2024-05-02,2024-05-02
...,...,...
ocd-bill/fff5d8af-80b1-4a98-a977-93fea7fea565,2023-01-10,2023-01-10
ocd-bill/fff7f8e7-1090-414c-b58f-da3d4dd4c1b7,2023-01-12,2023-01-12
ocd-bill/fff895cc-96a2-488a-b583-e73af4ae1bee,2024-02-09,2024-02-09


In [ ]:
actions_summary = bill_actions.groupby("bill_id").agg(
    first_action = ("date", "min"),
    last_action = ("date", "max"),
    num_actions = ("id", "count"),
    amendment_introduction = ("classification", lambda x: x.apply(lambda classif: "amendment-introduction" in classif).sum()),
    num_readings = ("classification", lambda x: x.apply(lambda classif: "reading" in classif).sum()),
    committee_passages = ("classification", lambda x: x.apply(lambda classif: "committee-passage" in classif).sum()),
    passed_one_plus_chamber = ("classification", lambda x: x.apply(lambda classif: bool(re.search(r"'passage'", classif))).any()),
    passed_legislature = ("classification", lambda x: x.apply(lambda classif: "executive-receipt" in classif).any()),
    became_law = ("classification", lambda x: x.apply(lambda classif: "became-law" in classif).any()),
    vetoed = ("classification", lambda x: x.apply(lambda classif: "veto" in classif).any())
    ).sort_values(by="num_actions",ascending=False)

In [ ]:
first_reading = bill_actions[bill_actions["classification"].apply(lambda x: "reading-1" in x)][["bill_id", "date"]]
first_reading.groupby("bill_id")["date"].min()

first_reading = first_reading.rename(columns = {"date": "first_reading_date"})

In [ ]:
first_referral = bill_actions[bill_actions["classification"].apply(lambda x: "referral-committee" in x)][["bill_id", "date"]]
first_referral.groupby("bill_id")["date"].min()

first_referral = first_referral.rename(columns = {"date": "first_referral_date"})

In [ ]:
first_committee_passage = bill_actions[bill_actions["classification"].apply(lambda x: "committee-passage" in x)][["bill_id", "date"]]
first_committee_passage.groupby("bill_id")["date"].min()

first_committee_passage = first_committee_passage.rename(columns = {"date": "committee_passage"})

In [ ]:
actions_summary = actions_summary.merge(first_reading, how = "inner", on = "bill_id")
actions_summary = actions_summary.merge(first_referral, how = "inner", on = "bill_id")
actions_summary = actions_summary.merge(first_committee_passage, how = "inner", on = "bill_id")

In [ ]:
actions_summary.head()

,bill_id,first_action,last_action,num_actions,amendment_introduction,num_readings,committee_passages,passed_one_plus_chamber,passed_legislature,became_law,vetoed,first_reading_date_x,first_referral_date_x,committee-passage,first_reading_date_y,first_referral_date_y,committee_passage
0,ocd-bill/1cfbb486-1332-45ed-8438-9bc174dd6a8d,2023-02-17,2023-08-11,187,5,11,1,True,True,True,False,2023-02-17,2023-02-28,2023-05-10,2023-02-17,2023-02-28,2023-05-10
1,ocd-bill/1cfbb486-1332-45ed-8438-9bc174dd6a8d,2023-02-17,2023-08-11,187,5,11,1,True,True,True,False,2023-02-17,2023-02-28,2023-05-10,2023-02-17,2023-05-16,2023-05-10
2,ocd-bill/1cfbb486-1332-45ed-8438-9bc174dd6a8d,2023-02-17,2023-08-11,187,5,11,1,True,True,True,False,2023-02-17,2023-02-28,2023-05-10,2023-05-11,2023-02-28,2023-05-10
3,ocd-bill/1cfbb486-1332-45ed-8438-9bc174dd6a8d,2023-02-17,2023-08-11,187,5,11,1,True,True,True,False,2023-02-17,2023-02-28,2023-05-10,2023-05-11,2023-05-16,2023-05-10
4,ocd-bill/1cfbb486-1332-45ed-8438-9bc174dd6a8d,2023-02-17,2023-08-11,187,5,11,1,True,True,True,False,2023-02-17,2023-02-28,2023-05-10,2023-05-11,2023-02-28,2023-05-10


In [ ]:
#actions_summary[actions_summary["passed_legislature"] == 0 and actions_summary["chambers_passed"] >= 2]

In [ ]:
actions_summary.head(5)

,bill_id,first_action,last_action,num_actions,amendment_introduction,num_readings,committee_passages,passed_one_plus_chamber,passed_legislature,became_law,vetoed,first_reading_date_x,first_referral_date_x,committee-passage,first_reading_date_y,first_referral_date_y,committee_passage
0,ocd-bill/1cfbb486-1332-45ed-8438-9bc174dd6a8d,2023-02-17,2023-08-11,187,5,11,1,True,True,True,False,2023-02-17,2023-02-28,2023-05-10,2023-02-17,2023-02-28,2023-05-10
1,ocd-bill/1cfbb486-1332-45ed-8438-9bc174dd6a8d,2023-02-17,2023-08-11,187,5,11,1,True,True,True,False,2023-02-17,2023-02-28,2023-05-10,2023-02-17,2023-05-16,2023-05-10
2,ocd-bill/1cfbb486-1332-45ed-8438-9bc174dd6a8d,2023-02-17,2023-08-11,187,5,11,1,True,True,True,False,2023-02-17,2023-02-28,2023-05-10,2023-05-11,2023-02-28,2023-05-10
3,ocd-bill/1cfbb486-1332-45ed-8438-9bc174dd6a8d,2023-02-17,2023-08-11,187,5,11,1,True,True,True,False,2023-02-17,2023-02-28,2023-05-10,2023-05-11,2023-05-16,2023-05-10
4,ocd-bill/1cfbb486-1332-45ed-8438-9bc174dd6a8d,2023-02-17,2023-08-11,187,5,11,1,True,True,True,False,2023-02-17,2023-02-28,2023-05-10,2023-05-11,2023-02-28,2023-05-10


In [ ]:
actions_summary["committee_passage"].head(20)

0    2023-05-10
1    2023-05-10
2    2023-05-10
3    2023-05-10
4    2023-05-10
5    2023-05-10
6    2023-05-10
7    2023-05-10
8    2023-05-10
9    2023-05-10
10   2023-05-10
11   2023-05-10
12   2023-05-10
13   2023-05-10
14   2023-05-10
15   2023-05-10
16   2023-05-10
17   2023-05-10
18   2023-05-10
19   2023-05-10
Name: committee_passage, dtype: datetime64[us]

In [ ]:
df = df.merge(actions_summary, how="inner", left_on="id", right_on="bill_id")

In [ ]:
df.head(5)

,id,identifier,title,classification,subject,session_identifier,jurisdiction,organization_classification,id_y,bill_id,abstract,note,first_action,last_action,num_actions,passed_legislature,became_law,vetoed
0,ocd-bill/f4c0d518-f592-4828-8c23-532ec44ed642,HB 1277,PEN CD-FELONY-SUSPEND BENEFITS,['bill'],[],103rd,Illinois,lower,014fdfc4-5501-4ad3-8e09-2148cc32aceb,ocd-bill/f4c0d518-f592-4828-8c23-532ec44ed642,Amends the General Provisions Article of the I...,NaN,2023-01-19,2024-06-25,7,False,False,False
1,ocd-bill/ea6533c0-942f-4f92-aef7-bef03046387c,HB 4017,$DHS-INDEPENDENT LIVING CNTRS,['bill'],[],103rd,Illinois,lower,e57822d9-cd63-48ab-a3df-f02d0d29d8bf,ocd-bill/ea6533c0-942f-4f92-aef7-bef03046387c,"Appropriates $16,358,900 from the General Reve...",NaN,2023-03-14,2024-03-15,8,False,False,False
2,ocd-bill/0372f02d-3773-4ce5-9f26-26f0141e2227,SB 2535,YOUTH NON-VIOLENT RESOURCES,['bill'],[],103rd,Illinois,upper,adb655ba-af0b-430d-a932-6f91043f3cac,ocd-bill/0372f02d-3773-4ce5-9f26-26f0141e2227,Amends the Illinois Criminal Justice Informati...,NaN,2023-03-07,2024-05-03,22,False,False,False
3,ocd-bill/90c3fdab-a08d-4e41-aac0-41fb9b462ea6,HR 376,CONGRATS-MARK AND ROSE FROSETH,['resolution'],[],103rd,Illinois,lower,b9907fdf-2639-4fe3-8d48-3e38a67b2550,ocd-bill/90c3fdab-a08d-4e41-aac0-41fb9b462ea6,Congratulates Mark and Rose Froseth for being ...,NaN,2023-08-10,2023-10-24,3,False,False,False
4,ocd-bill/d7335ae9-ecc0-41e9-8e89-94435aa00e59,HR 375,MEMORIAL-ANTHONY RUSSELL,['resolution'],[],103rd,Illinois,lower,b79c604a-9375-482a-89ea-cd92558efae6,ocd-bill/d7335ae9-ecc0-41e9-8e89-94435aa00e59,Mourns the death of Anthony Russell of Owings ...,NaN,2023-08-10,2023-10-24,3,False,False,False


In [ ]:
df[df["passed_legislature"] == True].head(5)

,id,identifier,title,classification,subject,session_identifier,jurisdiction,organization_classification,id_y,bill_id,abstract,note,first_action,last_action,num_actions,passed_legislature,became_law,vetoed
40,ocd-bill/58cd9efb-aec9-496c-a281-ef18a1940354,HB 2394,CERT SHORTHAND REPORT-VARIOUS,['bill'],[],103rd,Illinois,lower,2e25ef98-18d8-4cb9-b575-4e00b8f9b4ac,ocd-bill/58cd9efb-aec9-496c-a281-ef18a1940354,Amends the Regulatory Sunset Act. Repeals the ...,NaN,2023-02-14,2023-12-08,99,True,True,False
183,ocd-bill/80e388d4-c22e-456f-8695-dca2bce450cc,HB 1286,EQUITABLE RESTROOMS-ALL-GENDER,['bill'],[],103rd,Illinois,lower,42c84d1b-af10-4631-9e2b-71d33704618d,ocd-bill/80e388d4-c22e-456f-8695-dca2bce450cc,Amends the Equitable Restrooms Act. Provides t...,NaN,2023-01-19,2023-08-11,95,True,True,False
216,ocd-bill/2502440e-4b56-4f2e-8400-c1d72d91d496,HB 342,EDUCATION-TECH,['bill'],[],103rd,Illinois,lower,b25d8930-e377-46f0-9c56-593772519668,ocd-bill/2502440e-4b56-4f2e-8400-c1d72d91d496,Amends the School Code. Makes a technical chan...,NaN,2022-12-05,2023-08-03,50,True,True,False
262,ocd-bill/bf8e686b-34dd-470d-bd05-f35d8de15ac6,HB 2040,HWY CD-RESIDENCY OF CLERKS,['bill'],[],103rd,Illinois,lower,e264c61b-89cf-4825-a186-8fc8541e2a06,ocd-bill/bf8e686b-34dd-470d-bd05-f35d8de15ac6,Amends the Illinois Highway Code. Provides tha...,NaN,2023-02-01,2023-06-30,40,True,True,False
290,ocd-bill/3b169422-e3b5-4c53-8367-11aa86cb634d,HB 2285,DEFINE-HEALTH CARE EMPLOYER,['bill'],[],103rd,Illinois,lower,fa54fd87-7801-478e-a217-f238ad3eeac0,ocd-bill/3b169422-e3b5-4c53-8367-11aa86cb634d,Amends the Health Care Worker Background Check...,NaN,2023-02-10,2023-07-28,31,True,True,False


In [ ]:
df[df["title"].str.contains("GENDER NEUTRAL")]

,id,identifier,title,classification,subject,session_identifier,jurisdiction,organization_classification,id_y,bill_id,abstract,note,first_action,last_action,num_actions,passed_legislature,became_law,vetoed
930,ocd-bill/4d0b963b-abea-473c-8c3f-020a2aacb805,SB 2777,PROPERTY-GENDER NEUTRAL,['bill'],[],103rd,Illinois,upper,09da67e3-de74-4b43-bb20-8792abab870e,ocd-bill/4d0b963b-abea-473c-8c3f-020a2aacb805,Amends the Illinois Religious Freedom Protecti...,NaN,2024-01-17,2024-01-17,3,False,False,False
6425,ocd-bill/75a7eabd-aa28-488d-9a15-51e80f58fa2b,SJRCA 3,GENDER NEUTRAL-BILL OF RIGHTS,['constitutional amendment'],[],103rd,Illinois,upper,e3b5b0ec-faa0-49b5-98cb-c5ff37d310c4,ocd-bill/75a7eabd-aa28-488d-9a15-51e80f58fa2b,Proposes to amend the Bill of Rights Article o...,NaN,2023-02-28,2023-02-28,2,False,False,False
18153,ocd-bill/3e6a138b-8145-4d5f-bc35-cd72318ec08b,HB 45,GENDER NEUTRAL STATUTORY REF,['bill'],[],102nd,Illinois,lower,385c8844-fb1c-44aa-bbf7-2e0ddb64116f,ocd-bill/3e6a138b-8145-4d5f-bc35-cd72318ec08b,Amends various Acts and Codes. Changes all sta...,NaN,2021-01-13,2023-02-10,68,True,True,False
25804,ocd-bill/8eeae0e8-7e47-4262-b412-841dac6c65d4,SR 787,URGES-GENDER NEUTRAL ADS,['resolution'],[],102nd,Illinois,upper,9b1fdb00-2d4d-4584-bd56-19bd21054f4f,ocd-bill/8eeae0e8-7e47-4262-b412-841dac6c65d4,Urges the Federal Communication Commission to ...,NaN,2022-01-21,2023-01-10,8,False,False,False


## Merging Bill Sponsors

In [ ]:
bills_sponsorships.head()

,id,name,entity_type,organization_id,person_id,bill_id,primary,classification
0,26c646d8-96d9-4beb-a8f6-ea6a4db2fbc7,Don Harmon,person,NaN,ocd-person/15c56180-21e1-492d-874f-25d511f804ee,ocd-bill/fe963eaf-75a8-4583-a899-600743263f17,True,primary
1,b4d73e85-ea12-48d3-9b66-0525f1b6524e,Tom Weber,person,NaN,ocd-person/ba3dd226-a1c6-47af-917a-e12696d5aae5,ocd-bill/68da6087-7ed2-4c11-b9c5-15a054d9546a,True,primary
2,7bdda3ce-b118-4e6b-8f35-a08b3b198bcc,Laura Fine,person,NaN,ocd-person/0ce44c13-37aa-4560-8489-67c309da913a,ocd-bill/0808005e-cd25-4794-b7b0-c030ce2137cf,True,primary
3,bbc3bf0f-de33-4bc6-8186-c1440ce6b334,Don Harmon,person,NaN,ocd-person/15c56180-21e1-492d-874f-25d511f804ee,ocd-bill/90e509db-8d46-4b67-8cf5-71ef0ee34ad3,True,primary
4,7efa491c-a4f7-4c30-98ec-c907c6819988,Sue Rezin,person,NaN,ocd-person/a1ac69ba-49db-4529-926f-999e73ef9654,ocd-bill/0c26e5cf-326d-447c-bf69-e9cb8fc2d791,True,primary


In [ ]:
bills_sponsorships["entity_type"].unique()

<StringArray>
['person']
Length: 1, dtype: str

In [ ]:
bills_sponsorships["organization_id"].unique()

array([nan])

In [ ]:
bills_sponsorships.groupby("bill_id")["primary"].agg("sum").sort_values(ascending=False)

bill_id
ocd-bill/405dd593-58d5-4071-993e-61ac3619159b    2
ocd-bill/4038a1c2-93bf-4ae2-aef3-fe7dbdc26f61    2
ocd-bill/c9389219-4b60-4f9e-a4da-1fa11096c3c1    2
ocd-bill/3f78fa09-7047-41ef-be43-5fa50d1637d8    2
ocd-bill/c86c740d-c8c8-42f2-879d-005e5f80cbd7    2
                                                ..
ocd-bill/0cc1cbb8-6dde-4a00-947f-f131b2006bad    0
ocd-bill/0c7492a0-9570-40dc-b4ad-a4afb7b0fd2e    0
ocd-bill/087a1f56-01c8-4c40-9994-ea64a941cba5    0
ocd-bill/0642d199-7485-4d6b-8e15-dfba0540b91f    0
ocd-bill/023178a6-eddf-4fd9-88e9-a940ba52d475    0
Name: primary, Length: 26222, dtype: int64

In [ ]:
(bills_sponsorships.groupby("bill_id")["primary"].agg("sum") == 2).sum()  # 2,207 bills have 2 primary sponsors

np.int64(3116)

In [ ]:
sponsors_summary = bills_sponsorships.groupby("bill_id").agg(
    num_sponsors = ("id", "count"))

In [ ]:
primary_sponsors = bills_sponsorships["primary"] == True

In [ ]:
primary_sponsors_summary = bills_sponsorships[primary_sponsors].groupby("bill_id").agg(
    primary_sponsor_1 = ("name", "min"),
    primary_sponsor_2 = ("name", "max"))

primary_sponsors_summary["num_primary"] = 2

In [ ]:
# For bills with only 1 primary sponsor, clear out primary_sponsor_2
primary_sponsors_summary.loc[primary_sponsors_summary["primary_sponsor_2"] == primary_sponsors_summary["primary_sponsor_1"], "num_primary"] = 1
primary_sponsors_summary.loc[primary_sponsors_summary["primary_sponsor_2"] == primary_sponsors_summary["primary_sponsor_1"], "primary_sponsor_2"] = pd.NA

In [ ]:
primary_sponsors_summary["primary_sponsor_2"].count()

np.int64(3115)

In [ ]:
primary_sponsors_summary.head()

,primary_sponsor_1,primary_sponsor_2,num_primary
bill_id,,,
ocd-bill/00001671-dd8d-4867-9572-c8a250541072,Steve Stadelman,NaN,1
ocd-bill/0003dba0-925c-4f22-8e6c-2b9e10ed5c71,"Marcus C. Evans, Jr.",NaN,1
ocd-bill/000c89e5-2a87-470f-b478-601e14d57c5e,Charles Meier,Jason Plummer,2
ocd-bill/00107fd4-1658-4ff0-ace8-a4cc245cee80,David Koehler,NaN,1
ocd-bill/00120e78-5b0f-4d45-9705-abd27260421e,Dale Fowler,NaN,1


In [ ]:
# Join two sponsor tables
sponsors_summary = sponsors_summary.merge(primary_sponsors_summary, how="inner", left_on="bill_id", right_on="bill_id")
sponsors_summary.head()

,num_sponsors,primary_sponsor_1,primary_sponsor_2,num_primary
bill_id,,,,
ocd-bill/00001671-dd8d-4867-9572-c8a250541072,1,Steve Stadelman,NaN,1
ocd-bill/0003dba0-925c-4f22-8e6c-2b9e10ed5c71,1,"Marcus C. Evans, Jr.",NaN,1
ocd-bill/000c89e5-2a87-470f-b478-601e14d57c5e,2,Charles Meier,Jason Plummer,2
ocd-bill/00107fd4-1658-4ff0-ace8-a4cc245cee80,1,David Koehler,NaN,1
ocd-bill/00120e78-5b0f-4d45-9705-abd27260421e,1,Dale Fowler,NaN,1


In [ ]:
# Join full sponsors table with full bills table
df = df.merge(sponsors_summary, how="inner", left_on="id", right_on="bill_id")

In [ ]:
df = df.drop(columns = ["id_y", "bill_id"])

In [ ]:
# Adding an action timeframe column
df["first_to_last_action_time"] = df["last_action"] - df["first_action"]

In [ ]:
df.head()

,id,identifier,title,classification,subject,session_identifier,jurisdiction,organization_classification,abstract,note,first_action,last_action,num_actions,passed_legislature,became_law,vetoed,num_sponsors,primary_sponsor_1,primary_sponsor_2,num_primary,first_to_last_action_time
0,ocd-bill/f4c0d518-f592-4828-8c23-532ec44ed642,HB 1277,PEN CD-FELONY-SUSPEND BENEFITS,['bill'],[],103rd,Illinois,lower,Amends the General Provisions Article of the I...,NaN,2023-01-19,2024-06-25,7,False,False,False,5,Amy Elik,NaN,1,523 days
1,ocd-bill/ea6533c0-942f-4f92-aef7-bef03046387c,HB 4017,$DHS-INDEPENDENT LIVING CNTRS,['bill'],[],103rd,Illinois,lower,"Appropriates $16,358,900 from the General Reve...",NaN,2023-03-14,2024-03-15,8,False,False,False,6,Nabeela Syed,NaN,1,367 days
2,ocd-bill/0372f02d-3773-4ce5-9f26-26f0141e2227,SB 2535,YOUTH NON-VIOLENT RESOURCES,['bill'],[],103rd,Illinois,upper,Amends the Illinois Criminal Justice Informati...,NaN,2023-03-07,2024-05-03,22,False,False,False,12,"Elgie R. Sims, Jr.",NaN,1,423 days
3,ocd-bill/90c3fdab-a08d-4e41-aac0-41fb9b462ea6,HR 376,CONGRATS-MARK AND ROSE FROSETH,['resolution'],[],103rd,Illinois,lower,Congratulates Mark and Rose Froseth for being ...,NaN,2023-08-10,2023-10-24,3,False,False,False,1,Joyce Mason,NaN,1,75 days
4,ocd-bill/d7335ae9-ecc0-41e9-8e89-94435aa00e59,HR 375,MEMORIAL-ANTHONY RUSSELL,['resolution'],[],103rd,Illinois,lower,Mourns the death of Anthony Russell of Owings ...,NaN,2023-08-10,2023-10-24,3,False,False,False,1,Camille Y. Lilly,NaN,1,75 days


In [ ]:
df.columns

Index(['id', 'identifier', 'title', 'classification', 'subject',
       'session_identifier', 'jurisdiction', 'organization_classification',
       'abstract', 'note', 'first_action', 'last_action', 'num_actions',
       'passed_legislature', 'became_law', 'vetoed', 'num_sponsors',
       'primary_sponsor_1', 'primary_sponsor_2', 'num_primary',
       'first_to_last_action_time'],
      dtype='str')

In [ ]:
bill_votes.head()

,id,identifier,motion_text,motion_classification,start_date,result,organization_id,bill_id,bill_action_id,jurisdiction,session_identifier
0,ocd-vote/334da918-f592-4553-9588-9b4b03bbd550,NaN,Third Reading,[],2023-01-11,pass,ocd-organization/76120f9a-2982-4cc5-a6e2-de740...,ocd-bill/9d67a681-bda1-4a17-bdbc-1be2d2517c89,NaN,Illinois,103rd
1,ocd-vote/e343de4a-2336-4881-84db-2f5d90305508,NaN,Motion To Adopt,['passage'],2023-01-12,pass,ocd-organization/76120f9a-2982-4cc5-a6e2-de740...,ocd-bill/27f64059-0fae-42b1-9ef9-1e124c972329,NaN,Illinois,103rd
2,ocd-vote/87c7e156-12d0-4d4c-8ec7-1d3629a868cb,NaN,Motion To Adopt,['passage'],2023-01-25,pass,ocd-organization/76120f9a-2982-4cc5-a6e2-de740...,ocd-bill/45f93a22-7644-482e-be00-34ee33c1f07c,NaN,Illinois,103rd
3,ocd-vote/01b3799e-c3a9-4e6c-a0c0-c7863032f1d4,NaN,Third Reading,[],2023-02-01,pass,ocd-organization/76120f9a-2982-4cc5-a6e2-de740...,ocd-bill/6085d2e4-831c-4d58-b137-fea8b7d2b59c,NaN,Illinois,103rd
4,ocd-vote/0e7a1101-0e5e-4df4-a7be-aa67bdc8e6d9,NaN,Judiciary,[],2023-02-07,pass,ocd-organization/76120f9a-2982-4cc5-a6e2-de740...,ocd-bill/acfb3d4a-c582-4208-ba8c-bfa3587b3d3f,NaN,Illinois,103rd


In [ ]:
bill_votes["motion_text"].value_counts().head(20)

motion_text
Third Reading                        5860
Executive                            2737
Executive Appointments                573
Concurrence                           383
Judiciary                             233
Concurrence, Amendment 1              213
Concurrence, Amendment 2              207
Motion                                181
Insurance                             171
Public Health                         150
Education                             143
State Government Administration       142
Higher Education                      122
State Government                      121
Human Services                        114
Labor & Commerce                      108
Health and Human Services             107
Licensed Activities                    97
Energy & Environment                   96
Elem Sec Ed: Adm., Lic. & Charter      95
Name: count, dtype: int64

In [ ]:
df["first_to_last_action_time"].head(10)

0   523 days
1   367 days
2   423 days
3    75 days
4    75 days
5    21 days
6   635 days
7   161 days
8   510 days
9    19 days
Name: first_to_last_action_time, dtype: timedelta64[us]

In [ ]:
df.head()

,id,identifier,title,classification,subject,session_identifier,jurisdiction,organization_classification,id_y,bill_id,abstract,note
0,ocd-bill/f4c0d518-f592-4828-8c23-532ec44ed642,HB 1277,PEN CD-FELONY-SUSPEND BENEFITS,['bill'],[],103rd,Illinois,lower,014fdfc4-5501-4ad3-8e09-2148cc32aceb,ocd-bill/f4c0d518-f592-4828-8c23-532ec44ed642,Amends the General Provisions Article of the I...,NaN
1,ocd-bill/ea6533c0-942f-4f92-aef7-bef03046387c,HB 4017,$DHS-INDEPENDENT LIVING CNTRS,['bill'],[],103rd,Illinois,lower,e57822d9-cd63-48ab-a3df-f02d0d29d8bf,ocd-bill/ea6533c0-942f-4f92-aef7-bef03046387c,"Appropriates $16,358,900 from the General Reve...",NaN
2,ocd-bill/0372f02d-3773-4ce5-9f26-26f0141e2227,SB 2535,YOUTH NON-VIOLENT RESOURCES,['bill'],[],103rd,Illinois,upper,adb655ba-af0b-430d-a932-6f91043f3cac,ocd-bill/0372f02d-3773-4ce5-9f26-26f0141e2227,Amends the Illinois Criminal Justice Informati...,NaN
5,ocd-bill/b3f19d64-64ee-4061-94be-294fd4d5b6ca,SB 2816,ORGANIC WASTE COMPOSTING,['bill'],[],103rd,Illinois,upper,408e444f-c63f-4484-aacc-f63969f9f82d,ocd-bill/b3f19d64-64ee-4061-94be-294fd4d5b6ca,Amends the Environmental Protection Act. Requi...,NaN
6,ocd-bill/4ae56f70-b57b-4d05-a6ef-d22839d5fee1,HB 3418,REENTRY INTO THE WORKFORCE,['bill'],[],103rd,Illinois,lower,da4fc3ab-ebdf-428b-bc42-7fd2a25829c1,ocd-bill/4ae56f70-b57b-4d05-a6ef-d22839d5fee1,Creates the Securing All Futures through Equit...,NaN


In [ ]:
df.shape

(20039, 21)

In [ ]:
df["abstract_action"] = df["abstract"].str.split().str[0].str.lower()
#df["abstract_action"] = df["abstract_action"][0][0].lower()

In [ ]:
df["abstract_action"].head()

0          amends
1    appropriates
2          amends
5          amends
6         creates
Name: abstract_action, dtype: object

In [ ]:
df["abstract_action"].value_counts()

abstract_action
amends            16574
creates            1536
appropriates       1014
makes               696
repeals              84
provides             68
authorizes           29
if                   15
appropriations        8
restores              6
an                    5
reenacts              3
amends,               2
directs               2
replaces              1
establishes           1
sets                  1
removes               1
reappropriates        1
specifies             1
single                1
Name: count, dtype: int64

In [ ]:
df["abstract_action"].head(10)

0     amends
1     amends
2     amends
5     amends
6     amends
7     amends
8     amends
12    amends
13    amends
14    amends
Name: abstract_action, dtype: str

In [ ]:
df["abstract_action"].value_counts()

abstract_action
amends    20049
Name: count, dtype: int64

In [ ]:
df["abstract"].head()

0    Amends the General Provisions Article of the I...
1    Appropriates $16,358,900 from the General Reve...
2    Amends the Illinois Criminal Justice Informati...
5    Amends the Environmental Protection Act. Requi...
6    Creates the Securing All Futures through Equit...
Name: abstract, dtype: str

### Looking more into actions to find other milestones to add

In [ ]:
passed_legislature = df["passed_legislature"] == True
df[passed_legislature]["num_actions"].describe()  # The average bill that passes has 52 actions

count    2212.000000
mean       51.767631
std        23.225212
min        24.000000
25%        35.000000
50%        46.000000
75%        62.000000
max       187.000000
Name: num_actions, dtype: float64

In [ ]:
df[passed_legislature]["first_to_last_action_time"].describe()

count                        2212
mean     201 days 12:08:27.775768
std      114 days 16:43:53.665511
min              44 days 00:00:00
25%             143 days 00:00:00
50%             175 days 00:00:00
75%             185 days 00:00:00
max             765 days 00:00:00
Name: first_to_last_action_time, dtype: object

In [ ]:
df[passed_legislature]["num_sponsors"].describe()

count    2212.000000
mean       14.318716
std        15.118562
min         2.000000
25%         4.000000
50%         9.000000
75%        19.000000
max       134.000000
Name: num_sponsors, dtype: float64

In [ ]:
df[df["passed_legislature"] == True].head(5)

,id,identifier,title,classification,subject,session_identifier,jurisdiction,organization_classification,abstract,note,first_action,last_action,num_actions,passed_legislature,became_law,vetoed,num_sponsors,primary_sponsor_1,primary_sponsor_2,num_primary,first_to_last_action_time
40,ocd-bill/58cd9efb-aec9-496c-a281-ef18a1940354,HB 2394,CERT SHORTHAND REPORT-VARIOUS,['bill'],[],103rd,Illinois,lower,Amends the Regulatory Sunset Act. Repeals the ...,NaN,2023-02-14,2023-12-08,99,True,True,False,37,Bob Morgan,Suzy Glowiak Hilton,2,297 days
183,ocd-bill/80e388d4-c22e-456f-8695-dca2bce450cc,HB 1286,EQUITABLE RESTROOMS-ALL-GENDER,['bill'],[],103rd,Illinois,lower,Amends the Equitable Restrooms Act. Provides t...,NaN,2023-01-19,2023-08-11,95,True,True,False,46,Celina Villanueva,Katie Stuart,2,204 days
216,ocd-bill/2502440e-4b56-4f2e-8400-c1d72d91d496,HB 342,EDUCATION-TECH,['bill'],[],103rd,Illinois,lower,Amends the School Code. Makes a technical chan...,NaN,2022-12-05,2023-08-03,50,True,True,False,12,Carol Ammons,Kimberly A. Lightford,2,241 days
262,ocd-bill/bf8e686b-34dd-470d-bd05-f35d8de15ac6,HB 2040,HWY CD-RESIDENCY OF CLERKS,['bill'],[],103rd,Illinois,lower,Amends the Illinois Highway Code. Provides tha...,NaN,2023-02-01,2023-06-30,40,True,True,False,7,Adriane Johnson,"Curtis J. Tarver, II",2,149 days
290,ocd-bill/3b169422-e3b5-4c53-8367-11aa86cb634d,HB 2285,DEFINE-HEALTH CARE EMPLOYER,['bill'],[],103rd,Illinois,lower,Amends the Health Care Worker Background Check...,NaN,2023-02-10,2023-07-28,31,True,True,False,2,Adriane Johnson,Lakesia Collins,2,168 days


## Starting Analysis

In [ ]:
df.shape

(26167, 21)

In [ ]:
df.head()

,id,identifier,title,classification,subject,session_identifier,jurisdiction,organization_classification,abstract,note,first_action,last_action,num_actions,passed_legislature,became_law,vetoed,num_sponsors,primary_sponsor_1,primary_sponsor_2,num_primary,first_to_last_action_time
0,ocd-bill/f4c0d518-f592-4828-8c23-532ec44ed642,HB 1277,PEN CD-FELONY-SUSPEND BENEFITS,['bill'],[],103rd,Illinois,lower,Amends the General Provisions Article of the I...,NaN,2023-01-19,2024-06-25,7,False,False,False,5,Amy Elik,NaN,1,523 days
1,ocd-bill/ea6533c0-942f-4f92-aef7-bef03046387c,HB 4017,$DHS-INDEPENDENT LIVING CNTRS,['bill'],[],103rd,Illinois,lower,"Appropriates $16,358,900 from the General Reve...",NaN,2023-03-14,2024-03-15,8,False,False,False,6,Nabeela Syed,NaN,1,367 days
2,ocd-bill/0372f02d-3773-4ce5-9f26-26f0141e2227,SB 2535,YOUTH NON-VIOLENT RESOURCES,['bill'],[],103rd,Illinois,upper,Amends the Illinois Criminal Justice Informati...,NaN,2023-03-07,2024-05-03,22,False,False,False,12,"Elgie R. Sims, Jr.",NaN,1,423 days
3,ocd-bill/90c3fdab-a08d-4e41-aac0-41fb9b462ea6,HR 376,CONGRATS-MARK AND ROSE FROSETH,['resolution'],[],103rd,Illinois,lower,Congratulates Mark and Rose Froseth for being ...,NaN,2023-08-10,2023-10-24,3,False,False,False,1,Joyce Mason,NaN,1,75 days
4,ocd-bill/d7335ae9-ecc0-41e9-8e89-94435aa00e59,HR 375,MEMORIAL-ANTHONY RUSSELL,['resolution'],[],103rd,Illinois,lower,Mourns the death of Anthony Russell of Owings ...,NaN,2023-08-10,2023-10-24,3,False,False,False,1,Camille Y. Lilly,NaN,1,75 days


In [ ]:
df.classification.value_counts(normalize=True) * 100  # 76.4% of bills are classified as Bill (pre-filtering)

classification
['bill']    100.0
Name: proportion, dtype: float64

In [ ]:
df = df[df["classification"].apply(lambda x: "bill" in x)]

In [ ]:
df.passed_legislature.mean() * 100  # 8.77% of bills passed the legislature

np.float64(11.038474973801089)

In [ ]:
# TODO - Look into why no non-bills are passing. Need to use different mapping for actions?
df.groupby("classification")["passed_legislature"].agg("mean") * 100

classification
['bill']    11.038475
Name: passed_legislature, dtype: float64

In [ ]:
df.groupby("session_identifier")["passed_legislature"].agg(["mean", "count"])

,mean,count
session_identifier,,
102nd,0.113418,10113
103rd,0.107294,9926


In [ ]:
df.groupby("organization_classification")["passed_legislature"].agg(["mean", "count"])

,mean,count
organization_classification,,
lower,0.101764,11792
upper,0.122711,8247


In [ ]:
df.groupby(pd.cut(df["num_sponsors"], [0,1,  2, 5, 10, 15, 20, 25, 150]))["passed_legislature"].agg(["mean", "count"])

,mean,count
num_sponsors,,
"(0, 1]",0.000000,13024
"(1, 2]",0.113327,2056
"(2, 5]",0.291998,1887
"(5, 10]",0.358773,1271
"(10, 15]",0.458781,558
"(15, 20]",0.495146,412
"(20, 25]",0.569106,246
"(25, 150]",0.635897,585


In [ ]:
df.groupby("num_primary")["passed_legislature"].agg(["mean", "count"])

,mean,count
num_primary,,
1,0.001402,17114
2,0.748034,2925


In [ ]:
df.groupby(pd.cut(df["num_actions"], [0, 5, 10, 15, 20, 25, 30, 35, 45, 50, 200]))["passed_legislature"].agg(["mean", "count"])

,mean,count
num_actions,,
"(0, 5]",0.000000,5527
"(5, 10]",0.000000,8588
"(10, 15]",0.000000,1671
"(15, 20]",0.000000,715
"(20, 25]",0.071090,422
"(25, 30]",0.487941,539
"(30, 35]",0.626283,487
"(35, 45]",0.708029,685
"(45, 50]",0.777003,287


In [ ]:
df.groupby(df["first_action"].dt.month)["passed_legislature"].agg(["mean", "count"])

,mean,count
first_action,,
1,0.134620,4405
2,0.114780,13173
3,0.009494,316
4,0.024194,124
5,0.025974,154
6,0.000000,24
7,0.051282,39
8,0.059701,67
9,0.027027,74


In [ ]:
df.groupby("primary_sponsor_1")["passed_legislature"].agg(["mean", "count"]).sort_values("count", ascending = False)

,mean,count
primary_sponsor_1,,
Don Harmon,0.025854,2050
"Emanuel ""Chris"" Welch",0.000000,870
Emanuel Chris Welch,0.002497,801
Dan McConchie,0.010453,574
Tony M. McCombie,0.000000,420
...,...,...
John C. D'Amico,1.000000,3
Scott M. Bennett,1.000000,3
John Egofske,0.000000,3


In [ ]:
df.groupby("primary_sponsor_1")["passed_legislature"].agg(["mean", "count"]).sort_values("mean", ascending = False)

,mean,count
primary_sponsor_1,,
Antonio Muñoz,1.0,24
John Connor,1.0,16
Diane Pappas,1.0,1
John C. D'Amico,1.0,3
Melinda Bush,1.0,6
...,...,...
William Davis,0.0,31
William E Hauter,0.0,10
Willie Preston,0.0,40
